In [1]:
import pandas as pd

# Load data
df = pd.read_csv(
    "C:/Users/Azucar/Desktop/Master Thesis/Data/votes.csv")

# Rename the 6 perception variables
perception_map = {
    "livelier": "vitality",
    "more beautiful": "aesthetic",
    "safer": "safety",
    "more depressing": "depression",
    "wealthier": "wealth",
    "more boring": "boringness"}

df = df[df["study_question"].isin(perception_map.keys())].copy()
df["perception"] = df["study_question"].map(perception_map)

print("Remaining votes after perception filtering:", len(df))

# Convert pairwise votes into wins/losses
records = []
for _, row in df.iterrows():
    choice = str(row["choice"]).lower()
    if choice == "left":
        records.append([
            row["left"],
            row["perception"],
            1,
            row["lat_left"],
            row["long_left"],
            row["place_name_left"]])
        
        records.append([
            row["right"],
            row["perception"],
            0,
            row["lat_right"],
            row["long_right"],
            row["place_name_right"]])

    elif choice == "right":
        records.append([
            row["right"],
            row["perception"],
            1,
            row["lat_right"],
            row["long_right"],
            row["place_name_right"]])

        records.append([
            row["left"],
            row["perception"],
            0,
            row["lat_left"],
            row["long_left"],
            row["place_name_left"]])

votes = pd.DataFrame(
    records,
    columns=["image_id","perception","win","lat","lon","city"])
print("Votes converted:", len(votes))

# Compute wins and total votes
stats = votes.groupby(
    ["image_id","perception","lat","lon","city"]).agg(wins=("win","sum"),total=("win","count")).reset_index()
print("Unique image-perception pairs:", len(stats))

# Remove images with too few votes
stats = stats[stats["total"] >= 3].copy()
print("Remaining images after vote filtering:", len(stats))

# Compute Q scores
stats["Q"] = stats["wins"] / stats["total"]

# Invert boringness and depression perception
reverse_perceptions = ["depression", "boringness"]
stats.loc[stats["perception"].isin(reverse_perceptions),"Q"] = 1 - stats.loc[stats["perception"].isin(reverse_perceptions),"Q"]

# Binary labeling into 0/1
final_labels = []
for perception, group in stats.groupby("perception"):

    mean_Q = group["Q"].mean()
    std_Q  = group["Q"].std()

    Qhigh = mean_Q + std_Q
    Qlow  = mean_Q - std_Q

    g = group.copy()

    g["label"] = None
    g.loc[g["Q"] > Qhigh, "label"] = 1
    g.loc[g["Q"] < Qlow, "label"] = 0

    g = g.dropna(subset=["label"])

    final_labels.append(g)

final_labels = pd.concat(final_labels).reset_index(drop=True)
print("Samples after Q ± σ labeling:", len(final_labels))

# Filter to the 22 European cities
EUROPEAN_CITIES = [
    "Amsterdam","Barcelona","Berlin","Bratislava","Bucharest","Copenhagen","Dublin","Glasgow",
    "Helsinki","Kiev","Lisbon","London","Madrid","Milan","Moscow","Munich","Paris","Prague",
    "Rome","Stockholm","Vienna","Warsaw"]

final_labels = final_labels[
    final_labels["city"].isin(EUROPEAN_CITIES)].copy()
print("Final samples in European cities:", len(final_labels))

# Save the final dataset
output_path = "training_labels_Qsigma_EU_6perceptions.csv"
final_labels.to_csv(output_path, index=False)
print("Saved to:", output_path)
print(final_labels.head())
print("\nLabel distribution:")
print(final_labels.groupby(["perception","label"]).size())

Remaining votes after perception filtering: 1565540
Votes converted: 2718584
Unique image-perception pairs: 612747
Remaining images after vote filtering: 383113
Samples after Q ± σ labeling: 131231
Final samples in European cities: 42707
Done!
Saved to: training_labels_Qsigma_EU_6perceptions.csv
                    image_id perception        lat        lon        city  \
58  50e748e2d7c3df413b00139f  aesthetic  12.526316  55.713202  Copenhagen   
59  50e748e2d7c3df413b0013a5  aesthetic  12.543086  55.673530  Copenhagen   
60  50e748e2d7c3df413b0013a6  aesthetic  12.546335  55.704110  Copenhagen   
61  50e748e2d7c3df413b0013a8  aesthetic  12.552437  55.687702  Copenhagen   
62  50e748e3d7c3df413b0013b4  aesthetic  12.514635  55.708676  Copenhagen   

    wins  total         Q label  
58     4      4  1.000000     1  
59     4      4  1.000000     1  
60     1      6  0.166667     0  
61     4      5  0.800000     1  
62     0      3  0.000000     0  

Label distribution:
perception  lab